In [77]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries Loaded Successfully!")

✅ Libraries Loaded Successfully!


In [78]:
# CSV Load करें
matches = pd.read_csv('data/matches.csv')

# डेटा की झलक देखें
print("Total Matches:", len(matches))
print("\nColumns:", matches.columns.tolist())
print("\nपहले 5 मैच:")
matches.head()

Total Matches: 636

Columns: ['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

पहले 5 मैच:


,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,05-04-2017,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,06-04-2017,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,07-04-2017,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,2017,Indore,08-04-2017,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,2017,Bangalore,08-04-2017,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


In [79]:
# जिन मैचों का Winner है और DL Applied नहीं है, सिर्फ उन्हें रखें
filtered_matches = matches[(matches['winner'].notna()) & (matches['dl_applied'] == 0)].copy()
print(f"Clean Data के लिए उपलब्ध मैच: {len(filtered_matches)}")

# ड्रॉपडाउन में दिखाने के लिए Unique Teams और Venues निकालें (बाद में काम आएगा)
all_teams = sorted(set(filtered_matches['team1'].unique()).union(set(filtered_matches['team2'].unique())))
all_venues = sorted(filtered_matches['venue'].unique())
print("Unique Teams:", len(all_teams))
print("Unique Venues:", len(all_venues))

Clean Data के लिए उपलब्ध मैच: 617
Unique Teams: 14
Unique Venues: 35


In [80]:
# टीमों को नंबर देने के लिए Encoder
team_encoder = LabelEncoder()
all_teams_encoded = team_encoder.fit_transform(all_teams)

# एक नई DataFrame बनाएं जिसमें हम Features भरेंगे
features_df = filtered_matches[['team1', 'team2', 'venue', 'toss_winner', 'toss_decision', 'winner']].copy()

# 1. Venue को Encode करें
venue_encoder = LabelEncoder()
features_df['venue_encoded'] = venue_encoder.fit_transform(features_df['venue'])

# 2. Toss Decision को Encode करें (Bat = 0, Field = 1)
features_df['toss_decision_encoded'] = features_df['toss_decision'].map({'bat': 0, 'field': 1})

# 3. Toss Winner को Encode करें (अगर Team1 ने जीता तो 1, Team2 ने जीता तो 0)
features_df['toss_winner_encoded'] = (features_df['toss_winner'] == features_df['team1']).astype(int)

# 4. Head-to-Head Record (Team1 ने Team2 को कितनी बार हराया)
h2h_records = []
for _, row in features_df.iterrows():
    t1, t2 = row['team1'], row['team2']
    # इन दोनों टीमों के बीच पिछले सारे मैचों में Team1 की जीत
    past_matches = filtered_matches[
        ((filtered_matches['team1'] == t1) & (filtered_matches['team2'] == t2)) |
        ((filtered_matches['team1'] == t2) & (filtered_matches['team2'] == t1))
    ]
    if len(past_matches) > 0:
        wins = len(past_matches[past_matches['winner'] == t1])
        h2h = wins / len(past_matches)  # 0 से 1 के बीच Percentage
    else:
        h2h = 0.5  # पहली बार भिड़ रही हैं तो 50% मान लें
    h2h_records.append(h2h)

features_df['head_to_head'] = h2h_records

# 5. Recent Form (पिछले 5 मैचों में Team1 और Team2 की जीत %)
def get_recent_form(team_name, current_match_id):
    # उस टीम के पिछले मैच फिल्टर करें (current match से पहले के)
    past = filtered_matches[
        (filtered_matches['id'] < current_match_id) & 
        ((filtered_matches['team1'] == team_name) | (filtered_matches['team2'] == team_name))
    ].tail(5)  # सिर्फ पिछले 5 मैच
    
    if len(past) == 0:
        return 0.5  # नई टीम है तो 50%
    
    wins = (past['winner'] == team_name).sum()
    return wins / len(past)

form1_list = []
form2_list = []

for idx, row in features_df.iterrows():
    match_id = filtered_matches.loc[idx, 'id']
    form1 = get_recent_form(row['team1'], match_id)
    form2 = get_recent_form(row['team2'], match_id)
    form1_list.append(form1)
    form2_list.append(form2)

features_df['team1_form'] = form1_list
features_df['team2_form'] = form2_list

# 6. Team1 और Team2 को Encode करें (Label Encoding)
features_df['team1_encoded'] = team_encoder.transform(features_df['team1'])
features_df['team2_encoded'] = team_encoder.transform(features_df['team2'])

# 7. Target Variable (Label) बनाएं: अगर Team1 जीती तो 1, Team2 जीती तो 0
features_df['target'] = (features_df['winner'] == features_df['team1']).astype(int)

print("✅ Feature Engineering Complete!")
print("Total Features Created:", len(features_df.columns))
features_df.head()

✅ Feature Engineering Complete!
Total Features Created: 15


,team1,team2,venue,toss_winner,toss_decision,winner,venue_encoded,toss_decision_encoded,toss_winner_encoded,head_to_head,team1_form,team2_form,team1_encoded,team2_encoded,target
0,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,Sunrisers Hyderabad,23,1,0,0.666667,0.5,0.5,13,12,1
1,Mumbai Indians,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,field,Rising Pune Supergiant,16,1,0,0.250000,0.5,0.5,7,10,0
2,Gujarat Lions,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,field,Kolkata Knight Riders,25,1,0,0.750000,0.5,0.5,3,6,0
3,Rising Pune Supergiant,Kings XI Punjab,Holkar Cricket Stadium,Kings XI Punjab,field,Kings XI Punjab,11,1,0,0.500000,1.0,0.5,10,4,0
4,Royal Challengers Bangalore,Delhi Daredevils,M Chinnaswamy Stadium,Royal Challengers Bangalore,bat,Royal Challengers Bangalore,14,0,1,0.666667,0.0,0.5,12,2,1


In [81]:
# Model Train करने के लिए Final Features चुनें (सारे नंबर वाले कॉलम)
X = features_df[['venue_encoded', 'toss_decision_encoded', 'toss_winner_encoded', 
                 'head_to_head', 'team1_form', 'team2_form', 
                 'team1_encoded', 'team2_encoded']]
y = features_df['target']

# Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy Check
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")

🎯 Model Accuracy: 66.13%


In [82]:
# data/ फोल्डर में Model और Encoders को सेव करें
import joblib

# सुनिश्चित करें कि 'data' फोल्डर मौजूद है
import os
if not os.path.exists('data'):
    os.makedirs('data')

# Model और Encoders को सेव करें
joblib.dump(model, 'data/model.pkl')
joblib.dump(team_encoder, 'data/team_encoder.pkl')
joblib.dump(venue_encoder, 'data/venue_encoder.pkl')

print("✅ Model और Encoders 'data/' फोल्डर में सफलतापूर्वक सेव हो गए!")
print("📁 data/model.pkl बन गई है!")

✅ Model और Encoders 'data/' फोल्डर में सफलतापूर्वक सेव हो गए!
📁 data/model.pkl बन गई है!
